[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_6_8/11_tag_6_8_modelle_speichern_laden_custom_pytorch.ipynb)

# Tag 6-8 - 11 Modelle speichern und laden, auch mit Custom-Modulen

Speichern und Laden wird mit einem Custom-Modul gezeigt. Danach werden Vorhersagen als Bilder angezeigt und die geladene Version wird gegen den gespeicherten Checkpoint geprüft.


In [ ]:
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = False

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, Dataset, random_split

torch.manual_seed(RANDOM_STATE)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("CUDA verfügbar:", torch.cuda.is_available())
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

from pathlib import Path
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay

def accuracy_from_logits(logits, y):
    return (logits.argmax(dim=1) == y).float().mean().item()


def train_step(model, xb, yb, loss_fn, optimizer):
    model.train()
    xb, yb = xb.to(device), yb.to(device)
    optimizer.zero_grad()
    logits = model(xb)
    loss = loss_fn(logits, yb)
    loss.backward()
    optimizer.step()
    return loss.item(), accuracy_from_logits(logits.detach(), yb)


def evaluate(model, loader, loss_fn=None):
    model.eval()
    total_loss = 0.0
    correct = 0
    seen = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            if loss_fn is not None:
                total_loss += loss_fn(logits, yb).item() * len(xb)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            seen += len(xb)
    avg_loss = total_loss / seen if loss_fn is not None else np.nan
    return avg_loss, correct / seen


def train_epochs(model, train_loader, valid_loader, loss_fn, optimizer, epochs=5):
    history = []
    for epoch in range(epochs):
        losses = []
        accs = []
        for xb, yb in train_loader:
            loss, acc = train_step(model, xb, yb, loss_fn, optimizer)
            losses.append(loss)
            accs.append(acc)
        valid_loss, valid_acc = evaluate(model, valid_loader, loss_fn)
        row = {
            "epoch": epoch + 1,
            "train_loss": float(np.mean(losses)),
            "train_acc": float(np.mean(accs)),
            "valid_loss": float(valid_loss),
            "valid_acc": float(valid_acc),
        }
        history.append(row)
        print(
            f"Epoch {row['epoch']:02d}: "
            f"train_loss={row['train_loss']:.3f}, train_acc={row['train_acc']:.3f}, "
            f"valid_loss={row['valid_loss']:.3f}, valid_acc={row['valid_acc']:.3f}"
        )
    return history


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


## Custom-Modul und Modell


In [ ]:
class SquarePlus(nn.Module):
    def forward(self, x):
        return 0.5 * (x + torch.sqrt(x * x + 4))


class CustomCNN(nn.Module):
    """CNN mit der selbst definierten SquarePlus-Aktivierung."""
    def __init__(self, width=16, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, width, 3, padding=1),
            SquarePlus(),
            nn.MaxPool2d(2),
            nn.Conv2d(width, width * 2, 3, padding=1),
            SquarePlus(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Linear(width * 2, num_classes)

    def forward(self, x):
        x = self.features(x)
        return self.classifier(torch.flatten(x, start_dim=1))


class ReLUCNN(nn.Module):
    """Gleiche Architektur, aber mit der Standardaktivierung ReLU."""
    def __init__(self, width=16, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, width, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(width, width * 2, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Linear(width * 2, num_classes)

    def forward(self, x):
        x = self.features(x)
        return self.classifier(torch.flatten(x, start_dim=1))


## Training und bester Checkpoint


In [ ]:
digits = load_digits()
X = digits.images.astype(np.float32) / 16.0
y = digits.target.astype(np.int64)
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y)
train_data = TensorDataset(torch.tensor(X_train).unsqueeze(1), torch.tensor(y_train))
valid_data = TensorDataset(torch.tensor(X_valid).unsqueeze(1), torch.tensor(y_valid))

def make_loaders():
    # Beide Modellvarianten erhalten exakt dieselbe Batch-Reihenfolge.
    train_loader = DataLoader(
        train_data, batch_size=64, shuffle=True,
        generator=torch.Generator().manual_seed(RANDOM_STATE),
    )
    valid_loader = DataLoader(valid_data, batch_size=256)
    return train_loader, valid_loader


train_loader, valid_loader = make_loaders()
torch.manual_seed(RANDOM_STATE)
model = CustomCNN(width=12).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()
out_dir = Path("saved_models")
out_dir.mkdir(exist_ok=True)
checkpoint_path = out_dir / "custom_squareplus_cnn_digits.pt"
best_acc = -1.0
best_state = None
history_squareplus = []

# SquarePlus lernt auf diesem Beispiel deutlich langsamer.
for epoch in range(25):
    epoch_history = train_epochs(model, train_loader, valid_loader, loss_fn, optimizer, epochs=1)
    epoch_history[0]["epoch"] = epoch + 1
    history_squareplus.extend(epoch_history)
    valid_loss, valid_acc = evaluate(model, valid_loader, loss_fn)
    if valid_acc > best_acc:
        best_acc = valid_acc
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        torch.save({
            "model_state": best_state,
            "config": {"width": 12, "num_classes": 10},
            "classes": list(range(10)),
            "best_acc": best_acc,
        }, checkpoint_path)
print("SquarePlus – bester Checkpoint:", checkpoint_path, "best_acc=", round(best_acc, 3))


## Laden und identische Ausgaben prüfen


In [ ]:
checkpoint = torch.load(checkpoint_path, map_location=device)
loaded_a = CustomCNN(**checkpoint["config"]).to(device)
loaded_b = CustomCNN(**checkpoint["config"]).to(device)
loaded_a.load_state_dict(checkpoint["model_state"])
loaded_b.load_state_dict(checkpoint["model_state"])
loaded_a.eval()
loaded_b.eval()

xb, yb = next(iter(valid_loader))
with torch.no_grad():
    logits_a = loaded_a(xb.to(device)).cpu()
    logits_b = loaded_b(xb.to(device)).cpu()
print("Logits identisch:", torch.allclose(logits_a, logits_b))
print("Maximale Abweichung:", float((logits_a - logits_b).abs().max()))


## Vorhersagen als Bilder evaluieren


In [ ]:
probs = torch.softmax(logits_a, dim=1)
preds = probs.argmax(dim=1)
fig, axes = plt.subplots(2, 8, figsize=(12, 4))
for ax, img, true, pred, prob in zip(axes.ravel(), xb[:16], yb[:16], preds[:16], probs.max(dim=1).values[:16]):
    ax.imshow(img.squeeze(), cmap="gray", vmin=0, vmax=1)
    color = "green" if int(true) == int(pred) else "red"
    ax.set_title(f"t:{int(true)} p:{int(pred)}\n{float(prob):.2f}", color=color)
    ax.axis("off")
plt.tight_layout()

all_preds, all_true = [], []
with torch.no_grad():
    for xb_eval, yb_eval in valid_loader:
        all_preds.append(loaded_a(xb_eval.to(device)).argmax(dim=1).cpu())
        all_true.append(yb_eval)
ConfusionMatrixDisplay.from_predictions(torch.cat(all_true), torch.cat(all_preds), cmap="Blues")
plt.title("Geladenes SquarePlus-Modell: Confusion Matrix")
plt.grid(False);


## Variante 2 – ReLU: gleiche Architektur, schnelleres Lernen

Die Architektur, Daten, Initialisierung, Batch-Reihenfolge und der Optimizer bleiben gleich. Geändert wird ausschließlich die Aktivierung. Dadurch ist sichtbar, dass das SquarePlus-Modul hier sehr viel mehr Epochen benötigt, während ReLU in dieser Konfiguration deutlich schneller optimiert.


In [ ]:
relu_train_loader, relu_valid_loader = make_loaders()
torch.manual_seed(RANDOM_STATE)
relu_model = ReLUCNN(width=12).to(device)
relu_optimizer = torch.optim.Adam(relu_model.parameters(), lr=0.01)
relu_checkpoint_path = out_dir / "relu_cnn_digits.pt"
best_relu_acc = -1.0
history_relu = []

for epoch in range(15):
    epoch_history = train_epochs(
        relu_model, relu_train_loader, relu_valid_loader, loss_fn, relu_optimizer, epochs=1
    )
    epoch_history[0]["epoch"] = epoch + 1
    history_relu.extend(epoch_history)
    valid_loss, valid_acc = evaluate(relu_model, relu_valid_loader, loss_fn)
    if valid_acc > best_relu_acc:
        best_relu_acc = valid_acc
        best_relu_state = {k: v.detach().cpu().clone() for k, v in relu_model.state_dict().items()}
        torch.save({
            "model_state": best_relu_state,
            "config": {"width": 12, "num_classes": 10},
            "classes": list(range(10)),
            "best_acc": best_relu_acc,
        }, relu_checkpoint_path)

loaded_relu = ReLUCNN(width=12).to(device)
relu_checkpoint = torch.load(relu_checkpoint_path, map_location=device)
loaded_relu.load_state_dict(relu_checkpoint["model_state"])
relu_loss, relu_acc = evaluate(loaded_relu, relu_valid_loader, loss_fn)
print("ReLU – bester Checkpoint:", relu_checkpoint_path, "best_acc=", round(relu_acc, 3))


## Lernverlauf vergleichen

Die ReLU-Variante soll bereits nach wenigen Epochen klar über Zufall liegen. SquarePlus ist nicht fehlerhaft, braucht in dieser Konfiguration aber wesentlich länger.


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot([row["epoch"] for row in history_squareplus], [row["valid_acc"] for row in history_squareplus], label="SquarePlus (25 Epochen)")
plt.plot([row["epoch"] for row in history_relu], [row["valid_acc"] for row in history_relu], label="ReLU (15 Epochen)")
plt.axhline(0.1, color="gray", linestyle="--", label="Zufallsniveau (10 Klassen)")
plt.xlabel("Epoche")
plt.ylabel("Validierungs-Accuracy")
plt.title("SquarePlus vs. ReLU bei gleicher CNN-Architektur")
plt.legend()
plt.ylim(0, 1)
plt.show()

print(f"SquarePlus: {best_acc:.1%} nach 25 Epochen")
print(f"ReLU:       {relu_acc:.1%} nach maximal 15 Epochen")
